# Осенние продажи сети «Зерно»

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib
RANDOM_STATE=42
DATA=Path("data")
print(sys.version.split()[0],pd.__version__,np.__version__,matplotlib.__version__)

3.12.14 3.0.6 2.4.6 3.10.9


## 1. Цель

Директору нужны решения о сменах, лояльности и сезонном меню. Сравним прирост выручки с нулём по обоим переходам, лидера по местам с лидером по выручке, разницу чеков с порогом 5 %, пиковый час с утром выходных, разницу маржинальности с 3 п. п. Для сезонного напитка оценим долю и изменение чека.

**Вывод.** Для оценки эффекта меню запланируем отдельную проверку со сравнимыми кофейнями.

## 2. Данные

Одна строка выгрузки — один заказ осенью 2026 года. Справочники привязывают кофейню к городу и напиток к категории.

In [2]:

df=pd.read_csv(DATA / "orders_3m.csv",parse_dates=["datetime"])
shops=pd.read_csv(DATA / "shops.csv")
menu=pd.read_csv(DATA / "menu.csv")
full=df.merge(shops,on="shop_id",how="left",validate="many_to_one").merge(menu,on="drink",how="left",validate="many_to_one")
same_rows=len(full)==len(df)

In [3]:
print(df.shape,df.datetime.min(),df.datetime.max())
df.head()

(2089, 9) 2026-09-01 08:06:00 2026-11-30 21:39:00


,order_id,datetime,shop_id,drink,size,qty,amount,client_id,channel
0,O-00001,2026-09-01 10:31:00,PRM-1,Капучино,S,2,380,C1237,зал
1,O-00002,2026-09-01 13:34:00,PRM-1,Чай,S,2,240,C1074,с собой
2,O-00003,2026-09-01 20:42:00,PRM-1,Раф,M,1,325,NaN,с собой
3,O-00004,2026-09-01 19:55:00,PRM-1,Латте,M,1,262,C1153,доставка
4,O-00005,2026-09-01 19:32:00,PRM-1,Капучино,M,1,238,C1048,с собой


**Вывод.** В файле 2089 заказов и 9 полей. Время заказа позволяет сравнить три месяца.

## 3. Предобработка

In [4]:
print(df.isna().sum(),df.duplicated().sum())
assert same_rows and full.city.notna().all() and full.category.notna().all()

order_id       0
datetime       0
shop_id        0
drink          0
size           0
qty            0
amount         0
client_id    842
channel        0
dtype: int64 0


**Вывод.** Все заказы получили город и категорию. Заказы без карты сохранены для сравнения чеков. Когда единица анализа — человек, такие заказы исключаем.

## 4.1. H1. Рост сети

In [5]:
monthly=full.groupby(full.datetime.dt.to_period("M")).agg(revenue=("amount","sum"),orders=("order_id","count"),avg_check=("amount","mean"),clients=("client_id","nunique"))
monthly.avg_check=monthly.avg_check.round(1)
nov_vs_oct=round((monthly.revenue.iloc[-1]/monthly.revenue.iloc[-2]-1)*100,1)
monthly

,revenue,orders,avg_check,clients
datetime,,,,
2026-09,220241,651,338.3,201
2026-10,233756,700,333.9,208
2026-11,256198,738,347.2,214


**Вывод.** Выручка от сентября к ноябрю, ₽ — 220241, 233756, 256198. В ноябре прирост 9.6 % к октябрю. H1 подтвердилась.

## 4.2. H2. Кофейни и посадочные места

In [6]:
by_shop=full.groupby("shop_id").agg(city=("city","first"),revenue=("amount","sum"),seats=("seats","first"))
by_shop["per_seat"]=(by_shop.revenue/by_shop.seats).round()
by_shop=by_shop.sort_values("per_seat",ascending=False)
leader=by_shop.index[0]
leader_by_revenue=by_shop.revenue.idxmax()
by_shop

,city,revenue,seats,per_seat
shop_id,,,,
PRM-1,Пермь,186163,24,7757.0
KZN-1,Казань,181365,30,6046.0
KZN-2,Казань,98340,18,5463.0
TMN-1,Тюмень,96489,20,4824.0
PRM-2,Пермь,75377,16,4711.0
UFA-1,Уфа,72461,22,3294.0


**Вывод.** Лидер по выручке PRM-1, по размеру KZN-1. H2 не подтвердилась. На одно место больше всего заработала PRM-1.

## 4.3. H3. Сезонный напиток

In [7]:
nov=full[full.datetime.dt.month==11]
octo=full[full.datetime.dt.month==10]
seasonal_share=round(100*nov.loc[nov.drink=="Сезонный","amount"].sum()/nov.amount.sum(),1)
check_change=round(100*(nov.amount.mean()/octo.amount.mean()-1),1)
(seasonal_share,check_change)

(np.float64(15.6), np.float64(4.0))

**Вывод.** На сезонный напиток приходится 15.6 % выручки ноября, средний чек вырос на 4.0 %. Чтобы оценить собственный вклад напитка, нужен эксперимент.

## 4.4. H4. Средний чек с картой

In [8]:
g=full.groupby(full.client_id.notna()).amount.mean()
avg_card=round(g[True],1)
avg_no_card=round(g[False],1)
diff_pct=round(100*(g[True]/g[False]-1),1)
(avg_card,avg_no_card,diff_pct)

(np.float64(343.2), np.float64(335.2), np.float64(2.4))

**Вывод.** Чеки с картой и без — 343.2 и 335.2 ₽. Разница 2.4 % ниже порога 5 %. H4 не подтвердилась.

## 4.5. H5. Пиковая нагрузка

In [9]:
load=full.groupby([full.datetime.dt.weekday.rename("day"),full.datetime.dt.hour.rename("hour")]).size().unstack(fill_value=0)
peak=load.stack().idxmax()
load

hour,8,9,10,11,12,13,14,15,16,17,18,19,20,21
day,,,,,,,,,,,,,,
0,15,26,27,17,28,22,25,10,15,23,17,14,15,8
1,12,39,29,22,28,28,13,9,16,18,29,19,10,5
2,14,41,24,15,25,23,23,7,17,20,17,25,13,11
3,16,19,26,20,18,36,22,16,14,14,23,17,15,4
4,28,39,24,29,23,21,10,9,17,19,18,17,15,8
5,27,56,42,24,29,36,26,25,23,22,23,25,16,6
6,25,35,36,28,29,36,26,21,17,24,28,23,17,13


**Вывод.** Больше всего заказов (56) в день недели 5 (понедельник = 0), с 9:00. H5 подтвердилась.

## 4.6. H6. Маржинальность категорий

In [10]:
SIZE_K = {"S": 1.0, "M": 1.25, "L": 1.5}
cost=full.cost_s*full["size"].map(SIZE_K)*full.qty
v=full.groupby("category").amount.sum()
margin_raw=(1-cost.groupby(full.category).sum()/v)*100
margin=margin_raw.round(1)
margin

category
кофе       75.9
не кофе    77.3
dtype: float64

**Вывод.** Маржинальность кофе 75.9 %, другой категории 77.3 %. H6 не подтвердилась.

## 5. Выводы и рекомендации

In [11]:
def verdict(ok):
    return "подтвердилась" if ok else "не подтвердилась"
verdicts={"H1":verdict((monthly.revenue.diff().dropna()>0).all()),"H2":verdict(by_shop.seats.idxmax()==leader_by_revenue),"H4":verdict(g[True]/g[False]>=1.05),"H5":verdict(peak[0]>=5 and peak[1]<12),"H6":verdict(margin_raw["не кофе"]-margin_raw["кофе"]>=3)}
verdicts["H3"]="нужен эксперимент"
pd.Series(verdicts).sort_index()

H1        подтвердилась
H2     не подтвердилась
H3    нужен эксперимент
H4     не подтвердилась
H5        подтвердилась
H6     не подтвердилась
dtype: str

**Что сделали.** Сопоставили 2089 заказов со справочниками сети и проверили шесть гипотез.

**Что нашли.** Рост ноября 9.6 % совпал с ростом чека на 4.0 %. Разница чеков у заказов с картой и без ниже заданного порога.

**Что предлагаем.** Запускать новый напиток поэтапно и заранее назначить группы сравнения. План смен сверить с фактической нагрузкой по дням и часам.